# Chess.com Data Collection

This notebook collects public chess data from the Chess.com PubAPI.

The goal of this project is to build a personal chess analytics pipeline for:
- performance analysis
- rating progression
- opening analysis
- behavioral analytics
- visualization projects

Data source:
https://api.chess.com/pub/

## Imports

In [ ]:
import requests
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from datetime import datetime
import time

## User configuration

The Chess.com username is defined once in `src/config.py` and imported here.

In [ ]:
PROJECT_ROOT = Path().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import USERNAME

## Project paths

In [ ]:
RAW_DATA_DIR = PROJECT_ROOT / "dataset" / "raw" / USERNAME

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR

# API helper

Chess.com requires a valid User-Agent in order to avoid 403/rate limiting.

In [ ]:
headers = {
    "User-Agent": "chess-analytics-project (contact: elmurie@gmail.com)"
}

BASE_URL = f"https://api.chess.com/pub/player/{USERNAME}"


def get_json(url):

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Request failed: {url}")
        print(response.status_code)
        return None

    return response.json()

## Download player profile

In [ ]:
profile = get_json(BASE_URL)

profile_df = pd.json_normalize(profile)

profile_df.T.head(20)

In [ ]:
profile_df.to_csv(
    RAW_DATA_DIR / "profile.csv",
    index=False
)

## Download player stats

In [ ]:
stats = get_json(f"{BASE_URL}/stats")

stats_df = pd.json_normalize(stats)

stats_df.T.head(30)

In [ ]:
stats_df.to_csv(
    RAW_DATA_DIR / "stats.csv",
    index=False
)

## Game Archives

Chess.com stores games in monthly archives.

The API first returns a list of archive URLs,
then each archive contains the games played during that month.

In [ ]:
archives_data = get_json(
    f"{BASE_URL}/games/archives"
)

archives = archives_data["archives"]

len(archives)

## Download all games

In [ ]:
all_games = []

for archive in archives:

    data = get_json(archive)

    if not data:
        continue

    for game in data.get("games", []):

        all_games.append({

            "date": (
                datetime.fromtimestamp(
                    game.get("end_time")
                ).strftime("%Y-%m-%d %H:%M:%S")
                if game.get("end_time")
                else None
            ),

            "url": game.get("url"),

            "time_class": game.get("time_class"),
            "time_control": game.get("time_control"),

            "white": game.get("white", {}).get("username"),
            "black": game.get("black", {}).get("username"),

            "white_rating": game.get("white", {}).get("rating"),
            "black_rating": game.get("black", {}).get("rating"),

            "white_result": game.get("white", {}).get("result"),
            "black_result": game.get("black", {}).get("result"),

            "eco": game.get("eco"),

            "pgn": game.get("pgn")
        })

## Create Dataframe

In [ ]:
games_df = pd.DataFrame(all_games)

games_df.head()

## Dataset overview

The dataset looks pretty good!

In [ ]:
games_df.info()

In [ ]:
games_df.describe(include="all")

In [ ]:
games_df["time_class"].value_counts()

## Opponents list

We can extract the opponents' country, but we need to extract the profiles first

### Opponent normalisation 

In [ ]:
games_df["opponent"] = np.where(
    games_df["white"] == USERNAME,
    games_df["black"],
    games_df["white"]
)

games_df["opponent_clean"] = (
    games_df["opponent"]
    .dropna()
    .str.lower()
    .str.strip()
)

### Unique opponents

In [ ]:
opponents = (
    games_df["opponent_clean"]
    .dropna()
    .unique()
)

print(
    f"Unique opponents: {len(opponents)}"
)

### Save unique opponents

In [ ]:
opponents_df = pd.DataFrame({
    "opponent": opponents
})

opponents_path = (
    RAW_DATA_DIR
    / "unique_opponents.csv"
)

opponents_df.to_csv(
    opponents_path,
    index=False
)

print(
    f"Saved: {opponents_path}"
)

### Opponent enrichment cache

In [ ]:
profiles_path = (
    RAW_DATA_DIR
    / "opponent_profiles.csv"
)

### Load existing cache if present

In [ ]:
if profiles_path.exists():

    existing_profiles = pd.read_csv(
        profiles_path
    )

    print(
        f"Loaded existing profiles: {len(existing_profiles)}"
    )

else:

    existing_profiles = pd.DataFrame()

    print(
        "No existing profile cache found."
    )

### Already downloaded users

In [ ]:
if not existing_profiles.empty:

    downloaded_users = set(
        existing_profiles["opponent_clean"]
    )

else:

    downloaded_users = set()

### Safer get_json()

In [ ]:
def get_json(url):

    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        if response.status_code != 200:

            return None

        return response.json()

    except Exception as e:

        print(f"ERROR: {url}")

        print(e)

        return None

### Opponent profile enrichment

In [ ]:
new_profiles = []

total = len(opponents_df)

for i, opponent in enumerate(
    opponents_df["opponent"]
):

    # skip existing users
    if opponent in downloaded_users:

        continue

    try:

        OPPONENT_URL = (
            f"https://api.chess.com/pub/player/{opponent}"
        )

        profile = get_json(OPPONENT_URL)

        if not profile:

            continue

        country_url = profile.get("country")

        country_code = None

        if country_url:

            country_code = (
                country_url
                .split("/")[-1]
            )

        new_profiles.append({

            "opponent_clean": opponent,

            "country_code": country_code,

            "title": profile.get("title"),

            "followers": profile.get("followers"),

            "joined": profile.get("joined"),

            "last_online": profile.get("last_online")
        })

        # progress
        if i % 100 == 0:

            print(
                f"{i}/{total}"
            )

        # incremental save
        if i % 100 == 0:

            temp_df = pd.concat([

                existing_profiles,

                pd.DataFrame(new_profiles)

            ]).drop_duplicates(
                subset="opponent_clean"
            )

            temp_df.to_csv(
                profiles_path,
                index=False
            )

            print(
                f"Incremental save at {i}"
            )

        time.sleep(0.5)

    except Exception as e:

        print(
            f"ERROR on {opponent}"
        )

        print(e)

        continue

### Final save

In [ ]:
final_profiles = pd.concat([

    existing_profiles,

    pd.DataFrame(new_profiles)

]).drop_duplicates(
    subset="opponent_clean"
)

final_profiles.to_csv(
    profiles_path,
    index=False
)

print(
    f"Final profiles saved: {len(final_profiles)}"
)

## Save dataset

In [ ]:
games_df.to_csv(
    RAW_DATA_DIR / "games.csv",
    index=False
)

print("Dataset saved.")

# Next Steps

The next notebook will focus on:
- cleaning timestamps
- extracting player-side information
- parsing openings
- handling missing values
- feature engineering